[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/06_evaluation.ipynb)

# 06 — Final Evaluation

**Purpose.** Validate every configuration with Sionna-RT and compare baseline,
BO and MARL on identical terms. PROJECT.md sections 25, 26 and 27.

### Test-set discipline

The test split has been frozen since notebook 01. UE density used during
optimization came from the **training** split; the density here is rebuilt from
the **test** split, because the Band Priority Score weights locations by where
users were measured — scoring the optimized network on the same measurements
that shaped its objective is the leakage the split exists to prevent.

**Every number in this notebook comes from Sionna-RT.** A KPI improvement that
exists only in the surrogate is not a result (docs/adr/0003).

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same way
they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not ship.
Note that `data/` and `models/` are DVC-tracked and therefore *not* part of the
clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook. Forked the repository? Change these three values
# and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core"), ("sionna_rt", "sionna-rt")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not.
#
# The scene is the large one: data/external/simulation_map/ holds 3,753 meshes,
# so keep it on Drive rather than re-downloading it per session.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/band-tilt/data"
# CONFIG_OVERRIDES += [
#     f"data.mdt_path={DATA_ROOT}/raw/measurement_data.csv",
#     f"data.cell_config_path={DATA_ROOT}/raw/gcell_conf.csv",
#     f"data.scene_file={DATA_ROOT}/external/simulation_map/scene.xml",
#     f"data.train_path={DATA_ROOT}/processed/mdt_train.parquet",
#     f"data.test_path={DATA_ROOT}/processed/mdt_test.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook
starts the same way so that a cell copied between notebooks behaves identically.

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Load the three configurations

Baseline, BO, MARL. The baseline is not optional: every reported improvement is
stated relative to the current network, and a baseline evaluated in a different
run — against a different scene build or a different grid — is not a valid
reference.

In [ ]:
import json

from src.data.load import load_cell_config
from src.optim.space import TiltSpace
from src.radio import cell_band

table = cell_band.build_table(load_cell_config(cfg), cfg)
space = TiltSpace(table, cfg)

results_dir = Path("reports/results")
thetas = {"baseline": space.baseline()}
for method in ("bo", "marl"):
    path = results_dir / f"{method}_theta.json"
    thetas[method] = np.array(json.loads(path.read_text())["theta"])

pd.DataFrame(thetas, index=table["gcell_id"] + "|" + table["band"]).head(10)

## 3. Validate all three under one scene load

One scene, one grid, one density vector, three configurations. Reloading per
method costs minutes each and — worse — risks scoring the configurations against
subtly different geometry, which would make the comparison meaningless in a way
no number reveals.

In [ ]:
from src.evaluation import validate

results = validate.validate_many(thetas, cfg)
results

## 4. KPI comparison — PROJECT.md section 27.2

In the priority order from `cfg.kpi.order`, not alphabetically. The order *is*
the objective, and a table that hides it invites the reader to weigh weak rate
equally with hole rate.

The delta column is signed so that positive always means better — four KPIs
improve by decreasing and one by increasing, so an unlabelled delta gets
misread exactly once per reader.

In [ ]:
from src.evaluation import compare

kpi_table = compare.kpi_table(results, cfg)
kpi_table

## 5. Computational cost — PROJECT.md section 25.2

Sionna-RT evaluations separately from surrogate evaluations: they differ by
orders of magnitude, and a combined count makes the cheaper method look
expensive.

MARL's training cost belongs in this table. Excluding it because it is one-off is
an argument, not a measurement — and it only amortises if the policy transfers to
a network it was not trained on, which has to be demonstrated.

In [ ]:
compare.cost_table(results)

## 6. Stability across seeds — PROJECT.md section 25.3

Mean, standard deviation, **best and worst**. A method whose worst seed opens a
coverage hole is not usable regardless of its average, because a deployment gets
one run, not the mean of many.

State the seed count. Summary statistics over three runs and over thirty read
identically and mean very different things.

In [ ]:
runs = pd.read_csv(results_dir / "seed_runs.csv")
compare.stability_table(runs, cfg)

## 7. Tilt tables — PROJECT.md section 27.1

The deliverable a network engineer actually receives. The offset is derived here
purely to communicate how far each antenna moves; it was never optimised and
carries no penalty (docs/adr/0001).

Sorted by absolute offset: the rows that matter are the antennas that move most,
and the cell identifiers are opaque hashes that sort into no useful order.

In [ ]:
from src.evaluation import report

tilt_tables = {m: report.tilt_table(thetas[m], table) for m in ("bo", "marl")}
tilt_tables["bo"].head(15)

## 8. Spatial maps — PROJECT.md section 27.3

Where the numbers become an argument. The KPI table says hole rate fell three
points; these say whether the remaining holes moved to the edge of the area or
opened up where the users are.

The difference maps are the check for a bad trade: a configuration that improves
the aggregate by degrading a dense area is visible here and nowhere in section 4.

In [ ]:
from src.evaluation import analysis
from src.kpi import serving

for method in ("baseline", "bo", "marl"):
    analysis.rsrp_map(results[method]["rsrp"], cfg, title=f"{method} RSRP")
    analysis.coverage_map(results[method]["rsrp"], cfg)

analysis.difference_map(
    serving.max_rsrp(results["baseline"]["rsrp"]),
    serving.max_rsrp(results["bo"]["rsrp"]),
    cfg,
)
analysis.dominant_band_map(results["bo"]["dominant_band"], table, cfg)

## 9. Surrogate versus ground truth

How far the surrogate was off at the configurations that mattered. This closes
the loop on notebook 04: global error was measured there on sampled
configurations, and this is the error at the optimum each method actually
selected — the only place it can be measured.

A gap larger than the improvement being claimed means the result is not
supported, however good the ground-truth number looks alone.

In [ ]:
gaps = pd.DataFrame({m: results[m]["gap"] for m in ("bo", "marl")})
gaps["claimed_improvement_bo"] = kpi_table["bo_delta"]
gaps.loc[list(cfg.kpi.order)]

## 10. Export

In [ ]:
records = {
    m: report.summary(results[m], tilt_tables.get(m, pd.DataFrame()), cfg)
    for m in ("baseline", "bo", "marl")
}
report.export(records, "reports/results")
print("wrote reports/results/")

## 11. Conclusions

*Written by you, from the sections above. Replace every placeholder.*

**Which method produced the better network, and by how much.** *Under the
lexicographic priority, not a scalar score — say which KPI decided it.*

**What it cost.** *Sionna-RT evaluations and wall-clock for each, MARL training
included.*

**How stable each was.** *Across the seeds in section 6, worst case included.*

**Whether the surrogate held up.** *The gaps in section 9, against the
improvements claimed in section 4.*

**What the spatial maps show that the table does not.** *Section 8 — did the
optimized network trade a dense area for an empty one?*

**Limitations.** *Grid resolution, ray-tracing fidelity, the single observation
window, the band configuration in use, and anything the data could not support.*

**What would change the answer.** *The parameters from PROJECT.md section 30 that
were fixed by judgement rather than measurement.*